# 13 — Subtype feature extraction

Identical to notebook 08 but reads `labels.csv` from the subtype cache. Saved features feed the classical pipeline in notebook 14.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

from utils.seed import set_seed
from utils.data_histology import load_image_array
from utils.features_histology import extract_all_features

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
labels = pd.read_csv(cache_dir / 'labels.csv')
print('subtype samples:', len(labels))

In [ ]:
feature_names = None
rows = []
for path in tqdm(labels['path'].tolist(), desc='features'):
    img = load_image_array(path, image_size=cfg['data']['image_size'])
    vec, names = extract_all_features(img)
    if feature_names is None:
        feature_names = names
    rows.append(vec)
X = np.stack(rows).astype(np.float32)
y = labels['label_binary'].to_numpy().astype(int)
print('X', X.shape, 'features', len(feature_names))

In [ ]:
valid = ~np.any(~np.isfinite(X), axis=1)
print(f'valid rows: {valid.sum()}/{len(valid)}')
X = X[valid]
y = y[valid]
meta = labels.iloc[valid].reset_index(drop=True)

out = cache_dir / 'features.npz'
np.savez_compressed(out, X=X, y=y, feature_names=np.array(feature_names), path=meta['path'].to_numpy())
print('saved →', out)